In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/steam_games_cleaned.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (491, 8)


,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
0,Shadow Empire Deluxe 2049,2013-05-17,"0 .. 20,000",5.99,629.0,163.0,171.0,"Indie, Horror"
1,Crimson Rift Origins,2013-06-12,"100,000 .. 200,000",0.99,6293.0,23.0,NaN,Free To Play
2,Forgotten Legends,2012-04-14,"10,000,000 .. 20,000,000",14.99,690.0,76.0,152.0,Strategy
3,Last Empire Origins,2012-03-25,"20,000 .. 50,000",4.99,19009.0,486.0,43.0,"Puzzle, Sports"
4,Iron Tactics Deluxe 2,2009-05-24,"20,000 .. 50,000",59.99,3820.0,163.0,181.0,"Sports, Puzzle"


In [2]:
df["total_reviews"] = df["positive"] + df["negative"]

print(df[["positive", "negative", "total_reviews"]].head(10))

   positive  negative  total_reviews
0     629.0     163.0          792.0
1    6293.0      23.0         6316.0
2     690.0      76.0          766.0
3   19009.0     486.0        19495.0
4    3820.0     163.0         3983.0
5     289.0     116.0          405.0
6     785.0      46.0          831.0
7     923.0     722.0         1645.0
8       NaN     215.0            NaN
9    1025.0     824.0         1849.0


In [3]:
df["positive_rate"] = np.where(
    df["total_reviews"] > 0,
    (df["positive"] / df["total_reviews"]) * 100,
    np.nan
)

print(df[["positive", "total_reviews", "positive_rate"]].head(10))

   positive  total_reviews  positive_rate
0     629.0          792.0      79.419192
1    6293.0         6316.0      99.635845
2     690.0          766.0      90.078329
3   19009.0        19495.0      97.507053
4    3820.0         3983.0      95.907607
5     289.0          405.0      71.358025
6     785.0          831.0      94.464501
7     923.0         1645.0      56.109422
8       NaN            NaN            NaN
9    1025.0         1849.0      55.435370


In [4]:
df["negative_rate"] = np.where(
    df["total_reviews"] > 0,
    (df["negative"] / df["total_reviews"]) * 100,
    np.nan
)

print(df[["negative", "total_reviews", "negative_rate"]].head(10))

   negative  total_reviews  negative_rate
0     163.0          792.0      20.580808
1      23.0         6316.0       0.364155
2      76.0          766.0       9.921671
3     486.0        19495.0       2.492947
4     163.0         3983.0       4.092393
5     116.0          405.0      28.641975
6      46.0          831.0       5.535499
7     722.0         1645.0      43.890578
8     215.0            NaN            NaN
9     824.0         1849.0      44.564630


In [5]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = df["release_date"].dt.year

print(df[["name", "release_date", "release_year"]].head(10))

                        name release_date  release_year
0  Shadow Empire Deluxe 2049   2013-05-17        2013.0
1       Crimson Rift Origins   2013-06-12        2013.0
2          Forgotten Legends   2012-04-14        2012.0
3        Last Empire Origins   2012-03-25        2012.0
4      Iron Tactics Deluxe 2   2009-05-24        2009.0
5              Iron Survival   2007-08-24        2007.0
6       Cyber  Horizon  2049   2018-05-27        2018.0
7   Beyond Empire Remastered   2011-03-13        2011.0
8               Lost Odyssey   2013-07-31        2013.0
9      Silent Legends Deluxe   2020-04-24        2020.0


In [6]:
conditions = [
    df["price"].eq(0),
    df["price"].gt(0) & df["price"].le(10),
    df["price"].gt(10) & df["price"].le(30),
    df["price"].gt(30)
]

categories = [
    "Free",
    "Budget",
    "Mid-range",
    "Premium"
]

df["price_category"] = np.select(
    conditions,
    categories,
    default="Unknown"
)

print(df[["name", "price", "price_category"]].head(15))

                         name  price price_category
0   Shadow Empire Deluxe 2049   5.99         Budget
1        Crimson Rift Origins   0.99         Budget
2           Forgotten Legends  14.99      Mid-range
3         Last Empire Origins   4.99         Budget
4       Iron Tactics Deluxe 2  59.99        Premium
5               Iron Survival   9.99         Budget
6        Cyber  Horizon  2049  19.99      Mid-range
7    Beyond Empire Remastered  59.99        Premium
8                Lost Odyssey  13.49      Mid-range
9       Silent Legends Deluxe   1.79         Budget
10             Dark Souls III  49.99        Premium
11        Lost Journey Deluxe    NaN        Unknown
12        Last Tactics Deluxe  17.99      Mid-range
13      Iron Simulator Reborn   0.99         Budget
14     Last Chronicles Reborn   2.69         Budget


In [7]:
def calculate_owner_midpoint(value):

    if pd.isna(value):
        return np.nan

    # Extract numbers, including comma-separated numbers
    numbers = pd.Series(
        pd.Series([str(value)])
        .str.findall(r"\d[\d,]*")
        .iloc[0]
    ).str.replace(",", "", regex=False)

    numbers = pd.to_numeric(numbers, errors="coerce").dropna()

    if len(numbers) >= 2:
        return (numbers.iloc[0] + numbers.iloc[1]) / 2

    elif len(numbers) == 1:
        return numbers.iloc[0]

    return np.nan


df["estimated_owners_midpoint"] = (
    df["estimated_owners"].apply(calculate_owner_midpoint)
)

print(
    df[[
        "estimated_owners",
        "estimated_owners_midpoint"
    ]].head(15)
)

            estimated_owners  estimated_owners_midpoint
0                0 .. 20,000                    10000.0
1         100,000 .. 200,000                   150000.0
2   10,000,000 .. 20,000,000                 15000000.0
3           20,000 .. 50,000                    35000.0
4           20,000 .. 50,000                    35000.0
5                0 .. 20,000                    10000.0
6         200,000 .. 500,000                   350000.0
7     2,000,000 .. 5,000,000                  3500000.0
8         100,000 .. 200,000                   150000.0
9          50,000 .. 100,000                    75000.0
10      500,000 .. 1,000,000                   750000.0
11    2,000,000 .. 5,000,000                  3500000.0
12        100,000 .. 200,000                   150000.0
13               0 .. 20,000                    10000.0
14   5,000,000 .. 10,000,000                  7500000.0


In [8]:
print(df["estimated_owners_midpoint"].describe())

count    4.720000e+02
mean     2.845636e+06
std      4.439843e+06
min      1.000000e+04
25%      7.500000e+04
50%      7.500000e+05
75%      3.500000e+06
max      1.500000e+07
Name: estimated_owners_midpoint, dtype: float64


In [9]:
# Calculate popularity threshold
owner_median = df["estimated_owners_midpoint"].median()

print("Ownership Median:", owner_median)

# Identify games with sufficient information
valid_data = (
    df["positive_rate"].notna() &
    df["estimated_owners_midpoint"].notna() &
    df["total_reviews"].ge(20)
)

# Define high and low performance conditions
high_satisfaction = df["positive_rate"] >= 70

high_popularity = (
    df["estimated_owners_midpoint"] >= owner_median
)

conditions = [
    valid_data & high_satisfaction & high_popularity,

    valid_data & high_satisfaction & ~high_popularity,

    valid_data & ~high_satisfaction & high_popularity,

    valid_data & ~high_satisfaction & ~high_popularity
]

categories = [
    "Top Performer",
    "Hidden Gem",
    "Popular but Disliked",
    "Underperformer"
]

df["game_performance_category"] = np.select(
    conditions,
    categories,
    default="Insufficient Data"
)

print(
    df["game_performance_category"].value_counts()
)

Ownership Median: 750000.0
game_performance_category
Hidden Gem              186
Top Performer           182
Popular but Disliked     43
Insufficient Data        43
Underperformer           37
Name: count, dtype: int64


In [10]:
new_columns = [
    "total_reviews",
    "positive_rate",
    "negative_rate",
    "release_year",
    "price_category",
    "estimated_owners_midpoint",
    "game_performance_category"
]

print("New Analytical Columns:")
print(df[new_columns].head(20))

New Analytical Columns:
    total_reviews  positive_rate  negative_rate  release_year price_category  \
0           792.0      79.419192      20.580808        2013.0         Budget   
1          6316.0      99.635845       0.364155        2013.0         Budget   
2           766.0      90.078329       9.921671        2012.0      Mid-range   
3         19495.0      97.507053       2.492947        2012.0         Budget   
4          3983.0      95.907607       4.092393        2009.0        Premium   
5           405.0      71.358025      28.641975        2007.0         Budget   
6           831.0      94.464501       5.535499        2018.0      Mid-range   
7          1645.0      56.109422      43.890578        2011.0        Premium   
8             NaN            NaN            NaN        2013.0      Mid-range   
9          1849.0      55.435370      44.564630        2020.0         Budget   
10          610.0      58.196721      41.803279        2011.0        Premium   
11         6440.

In [11]:
print(df[new_columns].dtypes)

total_reviews                float64
positive_rate                float64
negative_rate                float64
release_year                 float64
price_category                object
estimated_owners_midpoint    float64
game_performance_category     object
dtype: object


In [12]:
print(df["game_performance_category"].value_counts())

game_performance_category
Hidden Gem              186
Top Performer           182
Popular but Disliked     43
Insufficient Data        43
Underperformer           37
Name: count, dtype: int64


In [13]:
print(df["price_category"].value_counts())

price_category
Budget       212
Mid-range    132
Premium       98
Free          34
Unknown       15
Name: count, dtype: int64


In [14]:
print(
    df[["positive_rate", "negative_rate"]].describe()
)

       positive_rate  negative_rate
count     466.000000     466.000000
mean       83.577482      16.422518
std        18.891527      18.891527
min         2.092050       0.098449
25%        76.766444       2.870360
50%        90.985449       9.014551
75%        97.129640      23.233556
max        99.901551      97.907950


In [15]:
df.to_csv(
    "../dataset/steam_games_enriched.csv",
    index=False
)

print("STEAM INSIGHT enriched dataset saved successfully!")

STEAM INSIGHT enriched dataset saved successfully!
